# Document-Aware Extraction with Llama Vision

Document-aware extraction notebook for processing invoices, receipts, and bank statements using Llama-3.2-Vision-Instruct.

**Key Features:**
- Document type detection - Automatically identifies invoice, receipt, or bank statement
- YAML-based prompt configuration - Static prompts for each document type
- V100 GPU optimization - ResilientGenerator with 6-tier OOM fallback system
- Universal field extraction - Consistent fields across all document types
- Memory management - Real-time fragmentation detection and cleanup
- Rich console interface - Beautiful formatting and comprehensive status reporting
- Production-ready error handling - Emergency model reload and CPU fallback
- Ground truth validation - Automated accuracy assessment and performance metrics

**Document Types Supported:**
- **Invoices** - Bills, quotes, estimates with line items and totals
- **Receipts** - Purchase confirmations with items and payment details
- **Bank Statements** - Transaction tables with dates and amounts

**Modern Architecture:**
- Modular V100-optimized extractor class
- External configuration files (YAML)
- Comprehensive memory monitoring
- Advanced error recovery systems

# Core Library Imports

Essential imports for Vision Language Model processing, image handling, and data manipulation.

In [1]:
# Standard library imports
import warnings
from pathlib import Path

# Third-party imports
import pandas as pd
import torch
import yaml
from PIL import Image
from rich import print as rprint
from rich.console import Console
from rich.syntax import Syntax
from rich.table import Table
from transformers import AutoProcessor, MllamaForConditionalGeneration

# Local imports - GPU optimization utilities
from common.gpu_optimization import (
    comprehensive_memory_cleanup,
    configure_cuda_memory_allocation,
    optimize_model_for_v100,
)

warnings.filterwarnings('ignore')

rprint("[bold green]✅ Core libraries imported successfully[/bold green]")

✅ Core libraries imported successfully

# Configuration and Model Setup

In [ ]:
# Configuration for document type detection and extraction
DETECTION_PROMPT_FILE = "prompts/document_type_detection.yaml"  # Document type detection prompts
DETECTION_PROMPT_KEY = "detection"  # Which detection prompt to use

# Extraction prompt files for each document type
PROMPT_FILES = {
    "INVOICE": "prompts/invoice_extraction.yaml",
    "RECEIPT": "prompts/receipt_extraction.yaml",
    "BANK_STATEMENT": "prompts/bank_statement_extraction.yaml"
}

# Default prompt key for each document type
PROMPT_KEYS = {
    "INVOICE": "standard",
    "RECEIPT": "standard", 
    "BANK_STATEMENT": "flat"
}

# Configuration for model
MODEL_PATH = "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct"

# Test images - can be any document type
TEST_IMAGE = "evaluation_data/commbank_flat_simple.png"  # Bank statement
# TEST_IMAGE = "evaluation_data/image_001.png"  # Invoice
# TEST_IMAGE = "evaluation_data/image_002.png"  # Receipt
# TEST_IMAGE = "evaluation_data/commbank_flat_complex.png"  # Bank statement
# TEST_IMAGE = "evaluation_data/anz_statement_001.png"  # Bank statement

# Ground truth CSV for evaluation
GROUND_TRUTH_CSV = "evaluation_data/ground_truth.csv"

# V100 PRODUCTION CONFIGURATION - MODERN APPROACH
USE_QUANTIZATION = True   # Enable BitsAndBytesConfig (modern approach)
DEVICE_MAP = "auto"       # Automatic device mapping
MAX_NEW_TOKENS = 4000     # V100 OPTIMIZED - Prevents OOM
TORCH_DTYPE = "bfloat16"  # V100 COMPATIBLE - More efficient
LOW_CPU_MEM_USAGE = True  # MEMORY EFFICIENT - Essential for V100

# Initialize Rich console
console = Console()
rprint("[bold blue]🚀 Document-Aware Configuration loaded[/bold blue]")
rprint("[yellow]⚠️ V100 Production Mode: Modern BitsAndBytesConfig approach[/yellow]")
rprint(f"[cyan]🎯 TEST IMAGE: {TEST_IMAGE}[/cyan]")
rprint("[green]💡 Document type will be auto-detected[/green]")

# Validate the selected image exists
if Path(TEST_IMAGE).exists():
    rprint(f"[green]✅ Test image found: {Path(TEST_IMAGE).name}[/green]")
else:
    rprint(f"[red]❌ Test image not found: {TEST_IMAGE}[/red]")
    rprint("[yellow]💡 Update TEST_IMAGE path above[/yellow]")

# Image Selection & Validation

Select and validate the bank statement image for processing, with fallback to available images.

In [3]:
# =============================================================================
# DOCUMENT IMAGE SELECTION & VALIDATION
# =============================================================================

from common.image_validator import validate_document_image

# Validate and display image information
IMAGE_PATH = validate_document_image(TEST_IMAGE)

🎯 Using document image selection...

🎉 Document ready: commbank_flat_simple.png

               📊 Document Image Information                
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property      ┃ Value                                    ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Filename      │ commbank_flat_simple.png                 │
│ Format        │ PNG                                      │
│ Dimensions    │ 600 × 800 pixels                         │
│ File Size     │ 32.0 KB                                  │
│ Full Path     │ evaluation_data/commbank_flat_simple.png │
│ Document Type │ To be detected...                        │
└───────────────┴──────────────────────────────────────────┘

─────────────────────────────────────────── Document Selection Complete ───────────────────────────────────────────

# Model Loading & Validation

Load the Llama 3.2 Vision model with comprehensive validation and GPU optimization for V100.

In [4]:
# =============================================================================
# MODEL LOADING & VALIDATION - V100 PRODUCTION OPTIMIZED
# =============================================================================

from common.model_loader import load_v100_model

# Load model with V100 optimizations
model, processor = load_v100_model(
    model_path=MODEL_PATH,
    use_quantization=USE_QUANTIZATION,
    device_map=DEVICE_MAP,
    max_new_tokens=MAX_NEW_TOKENS,
    torch_dtype=TORCH_DTYPE,
    low_cpu_mem_usage=LOW_CPU_MEM_USAGE
)

🚀 Loading Llama Vision model with V100 production optimizations...

🔧 Configuring V100-optimized CUDA memory allocation...

🔧 CUDA memory allocation configured: max_split_size_mb:64
💡 Using 64MB memory blocks to reduce fragmentation
📊 Initial CUDA state: Allocated=0.00GB, Reserved=0.00GB


🔧 Configuring V100-optimized 8-bit quantization with BitsAndBytesConfig

✅ V100-optimized BitsAndBytesConfig configured

...

✅ Model compatibility test passed

Performing initial V100 memory cleanup...

🧹 Clearing model caches...
✅ Model caches cleared
🧹 Memory state: Allocated=4.70GB, Reserved=4.75GB, Fragmentation=0.04GB


🎉 V100-optimized model loading and validation complete!

🔧 V100 optimizations active: CPU offload, vision skip, 32MB blocks

# Prompt Engineering

Define the optimized prompt for bank statement transaction extraction with clear visual guidance.

In [5]:
# Import V100-optimized modules 
from common.ground_truth_helpers import get_expected_transaction_count
from common.llama_vision_table_extractor import LlamaVisionTableExtractor

rprint("[bold green]✅ V100-optimized modules imported successfully[/bold green]")

# =============================================================================
# V100 PRODUCTION INITIALIZATION
# =============================================================================

# Initialize the V100-optimized extractor with existing model and processor
rprint("[bold green]🚀 Initializing V100-optimized LlamaVisionTableExtractor...[/bold green]")

try:
    extractor = LlamaVisionTableExtractor(processor=processor, model=model)
    
    # Validate V100 optimization status
    if hasattr(extractor, 'resilient_generator'):
        rprint("[green]✅ ResilientGenerator initialized for V100 OOM protection[/green]")
    else:
        rprint("[red]❌ ResilientGenerator initialization failed[/red]")
        
    rprint("[green]✅ V100-optimized extractor ready for production use![/green]")
    rprint("[cyan]🎯 V100 Features Active:[/cyan]")
    rprint("[cyan]   • ResilientGenerator with 6-tier OOM fallback[/cyan]")
    rprint("[cyan]   • Memory fragmentation detection and cleanup[/cyan]") 
    rprint("[cyan]   • Individual processing for memory stability[/cyan]")
    rprint("[cyan]   • Emergency model reload capabilities[/cyan]")
    rprint("[cyan]   • CPU fallback as ultimate safety net[/cyan]")
    
except Exception as e:
    rprint(f"[red]❌ CRITICAL: V100 extractor initialization failed: {e}[/red]")
    raise

✅ V100-optimized modules imported successfully

🚀 Initializing V100-optimized LlamaVisionTableExtractor...

✅ Using existing model and processor with V100 ResilientGenerator

✅ ResilientGenerator initialized for V100 OOM protection

✅ V100-optimized extractor ready for production use!

...

   • ResilientGenerator with 6-tier OOM fallback

   • Memory fragmentation detection and cleanup

   • Individual processing for memory stability

   • Emergency model reload capabilities

   • CPU fallback as ultimate safety net

# Document Type Detection

Detect the type of document (invoice, receipt, or bank statement) before selecting the appropriate extraction prompt.

In [6]:
# =============================================================================
# DOCUMENT TYPE DETECTION
# =============================================================================

from common.document_detector import detect_document_type

rprint("[bold blue]📋 Document Type Detection[/bold blue]")

# Detect document type using the extractor
DOCUMENT_TYPE = detect_document_type(
    image_path=IMAGE_PATH,
    extractor=extractor,
    detection_prompt_file=DETECTION_PROMPT_FILE,
    prompt_files=PROMPT_FILES,
    detection_prompt_key=DETECTION_PROMPT_KEY
)

console.rule("[bold green]Document Type Detection Complete[/bold green]")

# ============================================================================= 
# DOCUMENT-SPECIFIC PROMPT LOADING FROM YAML
# =============================================================================

from common.prompt_loader import load_document_prompt

rprint("[bold blue]📋 Loading Document-Specific Prompt from YAML[/bold blue]")

# Load document-specific prompt
EXTRACTION_PROMPT, prompt_name, prompt_description = load_document_prompt(
    prompt_files=PROMPT_FILES,
    prompt_keys=PROMPT_KEYS,
    document_type=DOCUMENT_TYPE
)

from common.ground_truth_helpers import get_expected_transaction_count
from common.llama_vision_table_extractor import LlamaVisionTableExtractor

rprint("[bold green]✅ V100-optimized modules imported successfully[/bold green]")


📋 Document Type Detection

✅ Loaded detection prompt: Document Type Detection

🔍 Detecting document type for: commbank_flat_simple.png

🧪 RUNNING V100-OPTIMIZED EXTRACTION TEST

📄 Image: evaluation_data/commbank_flat_simple.png

...

❌ Prompt file not found at prompts/bank_statement_extraction_original.yaml

💡 Please ensure the YAML file exists at: 
/home/jovyan/nfs_share/tod/LMM_POC/prompts/bank_statement_extraction_original.yaml

❌ CRITICAL: Prompt file not found: prompts/bank_statement_extraction_original.yaml

Please create the prompt YAML file before running.

FileNotFoundError: Required prompt file not found: prompts/bank_statement_extraction_original.yaml

# Document-Aware Extraction

In [ ]:
# =============================================================================
# V100 PRODUCTION DEMONSTRATION - DOCUMENT-AWARE EXTRACTION
# =============================================================================

from common.result_display import display_extraction_test_results
from common.ground_truth_helpers import get_expected_transaction_count

# Test with the current image and document-specific prompt
if IMAGE_PATH:
    # Use the enhanced test_extraction method which includes V100 memory monitoring
    test_result = extractor.test_extraction(IMAGE_PATH, EXTRACTION_PROMPT)
    
    # Display comprehensive test results using the result display module
    display_extraction_test_results(
        test_result=test_result,
        document_type=DOCUMENT_TYPE,
        image_path=IMAGE_PATH,
        expected_count_func=get_expected_transaction_count,
        ground_truth_csv=GROUND_TRUTH_CSV
    )
else:
    rprint("[red]❌ No image path available for V100 testing[/red]")

# Visual Comaprison

In [ ]:
# =============================================================================
# VISUAL COMPARISON - DOCUMENT IMAGE DISPLAY
# =============================================================================

from IPython.display import Image as IPImage
from IPython.display import display

if IMAGE_PATH and Path(IMAGE_PATH).exists():
    rprint(f"[cyan]📄 Displaying {DOCUMENT_TYPE} for visual comparison: {Path(IMAGE_PATH).name}[/cyan]")
    
    # Display the image
    display(IPImage(filename=IMAGE_PATH))
    
    rprint(f"[green]✅ Compare extraction results above with the visual {DOCUMENT_TYPE} image[/green]")
else:
    rprint(f"[red]❌ Document image not found: {IMAGE_PATH}[/red]")

# Field-Level Ground Truth Evaluation

Comprehensive field-by-field comparison between extracted data and ground truth values.

In [ ]:
# =============================================================================
# RESPONSE PREPROCESSING - Import from common module
# =============================================================================

from common.response_preprocessing import (
    clean_markdown_response,
    extract_statement_date_range,
    extract_transaction_data_from_table,
    map_bank_fields_to_universal,
    map_invoice_fields_to_universal,
    map_receipt_fields_to_universal,
    map_fields_to_universal,  # Universal mapping function
)

rprint("[green]✅ Response preprocessing functions imported from common module[/green]")

# Ground Truth Evaluation Summary 

In [ ]:
# =============================================================================
# FIELD-LEVEL GROUND TRUTH EVALUATION - DOCUMENT-AWARE
# =============================================================================

from common.ground_truth_evaluator import GroundTruthEvaluator

if IMAGE_PATH and 'test_result' in globals() and test_result.get('success'):
    # Perform comprehensive ground truth evaluation using the new module
    evaluator = GroundTruthEvaluator(GROUND_TRUTH_CSV)
    evaluation_results = evaluator.evaluate_extraction(
        test_result=test_result,
        document_type=DOCUMENT_TYPE,
        image_path=IMAGE_PATH
    )
else:
    rprint("[yellow]⚠️ No extraction results available for evaluation[/yellow]")
    rprint("[dim]Run the extraction test in the cell above first[/dim]")